# 09 — Phase 2 Submission (Standalone Pipeline)

Self-contained notebook for the Phase 2 submission of the Information Retrieval project.
Pipeline: TF-IDF + LinearSVC → Query Expansion → PRF → Hybrid BM25 + Embeddings (RRF) → Hard-Filter Reranking

**Compatible:** Local · Google Colab · Kaggle  
**No dependency on `src/`** — all required code is copied inline.

## Optional Dependency Installation

The cell below auto-installs required packages on Kaggle and Colab. Run it manually if needed in other environments.

In [ ]:
# Install missing dependencies (runs automatically on Kaggle / Colab)
import os, subprocess, sys

_need_install = (
    os.path.exists("/kaggle/working")          # Kaggle
    or os.path.exists("/content")              # Colab
    or "google.colab" in sys.modules
)
if _need_install:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "numpy", "pandas", "scikit-learn", "rank-bm25",
        "sentence-transformers", "tqdm",
    ])

## Shared Imports

In [ ]:
from __future__ import annotations
import json, hashlib, os, sys, re, time, random
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Literal, Sequence, Tuple, TYPE_CHECKING

import numpy as np
from numpy.typing import ArrayLike
from tqdm import tqdm
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

if TYPE_CHECKING:
    from sentence_transformers import SentenceTransformer

## Runtime Setup

This notebook auto-detects the execution environment:

- **Kaggle**: data in `/kaggle/input/`, outputs in `/kaggle/working/`
- **Google Colab**: standalone mode with uploaded files or mounted Drive
- **Local**: standard project layout `data/raw/`, `data/cache/`, `outputs/submissions/`

In [ ]:
cwd = Path.cwd().resolve()
IS_KAGGLE = os.path.exists("/kaggle/working")
IS_COLAB  = "google.colab" in sys.modules or os.path.exists("/content")

def _has_files(base, filenames):
    return all((Path(base) / name).exists() for name in filenames)

_RAW_NEEDED  = ["docs.json", "queries_test.json", "queries_train.json"]

if IS_KAGGLE:
    # Data in /kaggle/input/, output in /kaggle/working/
    _input_root = Path("/kaggle/input")
    _kaggle_data = None
    # Try canonical competition path first
    _canonical = _input_root / "competitions" / "retrieval-engine-competition"
    if _canonical.exists() and _has_files(_canonical, _RAW_NEEDED):
        _kaggle_data = _canonical
    else:
        for _p in sorted(_input_root.glob("*")):
            if _p.is_dir() and _has_files(_p, _RAW_NEEDED):
                _kaggle_data = _p
                break
    if _kaggle_data is None:
        raise FileNotFoundError("Kaggle: no /kaggle/input/* folder containing the data was found.")
    RAW_DIR           = _kaggle_data
    CACHEDIR          = Path("/kaggle/working/cache")
    OUTPUT_SUBMISSION = Path("/kaggle/working/submission.csv")
    RUNTIME_MODE      = "kaggle"

else:
    # Local or Colab: standalone-first, then project-mode
    def _find_project_root(start):
        for candidate in (start, *start.parents):
            if (candidate / "src").is_dir() and (candidate / "data").is_dir():
                return candidate
        return None

    if _has_files(cwd, _RAW_NEEDED):
        RAW_DIR           = cwd
        CACHEDIR          = cwd / "cache"
        OUTPUT_SUBMISSION = cwd / "submission.csv"
        RUNTIME_MODE      = "colab_standalone" if IS_COLAB else "local_standalone"

    elif _has_files(cwd / "raw", _RAW_NEEDED):
        RAW_DIR           = cwd / "raw"
        CACHEDIR          = cwd / "cache"
        OUTPUT_SUBMISSION = cwd / "submission.csv"
        RUNTIME_MODE      = "colab_subdirs" if IS_COLAB else "local_subdirs"

    else:
        _root = _find_project_root(cwd)
        if _root is None:
            raise FileNotFoundError(
                f"Could not locate data from cwd={cwd}.\n"
                "Place docs.json / queries_*.json next to the notebook (standalone mode),\n"
                "dans un sous-dossier raw/, ou lancez depuis la racine du projet."
            )
        RAW_DIR           = _root / "data" / "raw"
        CACHEDIR          = _root / "data" / "cache"
        OUTPUT_SUBMISSION = _root / "outputs" / "submissions" / "submission.csv"
        RUNTIME_MODE      = "colab_project" if IS_COLAB else "local_project"

NOTEBOOK_T0 = time.perf_counter()
print(f"runtime_mode      : {RUNTIME_MODE}")
print(f"RAW_DIR           : {RAW_DIR}")
print(f"CACHEDIR          : {CACHEDIR}")
print(f"OUTPUT_SUBMISSION : {OUTPUT_SUBMISSION}")

## Environment / Versions

In [ ]:
import platform
import sklearn

print(f"Python           : {platform.python_version()}")
print(f"numpy            : {np.__version__}")
print(f"pandas           : {pd.__version__}")
print(f"scikit-learn     : {sklearn.__version__}")

try:
    import sentence_transformers
    print(f"sentence-transformers : {sentence_transformers.__version__}")
except ImportError:
    print("sentence-transformers : NOT INSTALLED")

try:
    import rank_bm25
    print(f"rank-bm25        : {rank_bm25.__version__}")
except (ImportError, AttributeError):
    try:
        import rank_bm25
        print("rank-bm25        : installed (version unknown)")
    except ImportError:
        print("rank-bm25        : NOT INSTALLED")

## Phase 2 Configuration

Edit the constants below to adjust the pipeline behaviour.

In [ ]:
RANDOM_SEED = 42

# --- Classification ---
CLASSIFIER_METHOD = 'svc'
FEATURE_METHOD    = 'tfidf'

# --- Reranking ---
RERANK_STRATEGY     = 'hard_filter'
RERANK_BOOST_FACTOR = 0.5

# --- Retrieval ---
TOP_K              = 100
TEXT_FIELD         = 'content'
RETRIEVAL_K        = TOP_K * 10   # 1000 candidates
PRF_TOP_K          = 3
PRF_MAX_TAGS       = 10

# --- Embeddings ---
EMBEDDING_MODEL_NAME = 'all-MiniLM-L12-v2'

# --- Hybrid BM25 + Embeddings ---
HYBRID_RRF_K             = 30
HYBRID_WEIGHT_EMBEDDINGS = 3.5
HYBRID_WEIGHT_BM25       = 0.5

# --- Validation ---
if TOP_K != 100:
    raise ValueError(f"Kaggle submission requires TOP_K=100, got {TOP_K}.")
if RERANK_STRATEGY == 'soft_boost' and CLASSIFIER_METHOD == 'svc':
    print('[warn] soft_boost requires predict_proba — use CLASSIFIER_METHOD=logreg.')

## JSON Utilities

Verbatim copy of `src/data/load.py`

In [ ]:
# === Verbatim copy of src/data/load.py ===

def load_json(path: Path) -> object:
    """
    Load a JSON file and return the parsed Python object.

    Parameters
    ----------
    path : Path
        Path to the JSON file.

    Returns
    -------
    object
        Parsed JSON content.
    """
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

## Text Preprocessing

Verbatim copy of `src/data/preprocess.py`

In [ ]:
# === Verbatim copy of src/data/preprocess.py ===

def to_string(v):
    """Return a safe string representation of a value."""
    if v is None:
        return ""
    return str(v).replace("\ufffd", " ").strip()


def build_content(item: dict) -> str:
    """
    Build the merged text representation used by retrieval and classification.

    Parameters
    ----------
    item : dict
        Record that may contain `title`, `text`, and `tags`.

    Returns
    -------
    str
        Concatenated text content.
    """
    parts = []

    title = to_string(item.get("title"))
    if title:
        parts.append(title)

    text = to_string(item.get("text"))
    if text:
        parts.append(text)

    tags = item.get("tags")

    if isinstance(tags, list):
        valid_tags = [to_string(t) for t in tags if to_string(t)]
        if valid_tags:
            parts.append(" ".join(valid_tags))
    else:
        tag_str = to_string(tags)
        if tag_str:
            parts.append(tag_str)

    return " ".join(parts)


def clean_text(text: str) -> str:
    """
    Apply light normalization to a text string.
    """
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"([!?.]){2,}", r"\1", text)
    return text.strip()


def process_items(items: list[dict], clean: bool) -> list[dict]:
    """Copy items, add `content`, and optionally normalize it."""
    enriched_items = []
    for item in items:
        new_item = item.copy()
        content = build_content(new_item)
        if clean:
            content = clean_text(content)

        new_item["content"] = content
        enriched_items.append(new_item)
    return enriched_items


def add_content_field(docs: list[dict], queries: list[dict], clean: bool) -> tuple[list[dict], list[dict]]:
    """
    Add a `content` field to documents and queries.

    Parameters
    ----------
    docs : list[dict]
        Document records.
    queries : list[dict]
        Query records.
    clean : bool
        Whether to apply `clean_text` to the generated content.

    Returns
    -------
    tuple[list[dict], list[dict]]
        Enriched documents and queries.
    """
    docs_enriched = process_items(docs, clean)
    queries_enriched = process_items(queries, clean)

    return docs_enriched, queries_enriched

## BM25

Verbatim copy of `src/retrieval/bm25.py`

In [ ]:
# === Verbatim copy of src/retrieval/bm25.py ===

from rank_bm25 import BM25Okapi, BM25Plus

_TECH_TOKEN_RE = re.compile(
    r"(?:\.[a-z0-9]+|[a-z0-9]+(?:[._-][a-z0-9]+)*(?:\+\+|#)?)"
)


def tokenize(text: str) -> List[str]:
    """
    Tokenize text while preserving common technical tokens.
    """
    return _TECH_TOKEN_RE.findall(text.lower())


def fit_bm25(
    docs: List[Dict],
    text_field: str = "content",
    method: str = "plus",
) -> object:
    """
    Fit a BM25 model on the document corpus.

    Parameters
    ----------
    docs : list[dict]
        Documents to index.
    text_field : str, default="content"
        Field used as input text.
    method : str, default="plus"
        BM25 variant: `"plus"` or `"okapi"`.

    Returns
    -------
    object
        Fitted BM25 model.
    """
    texts = [doc[text_field] for doc in docs]
    tokenized_texts = [tokenize(text) for text in texts]

    if method == "plus":
        bm25_model = BM25Plus(tokenized_texts)
    elif method == "okapi":
        bm25_model = BM25Okapi(tokenized_texts)
    else:
        raise ValueError(f"method must be 'plus' or 'okapi', not {method!r}")

    return bm25_model


def retrieve_bm25(
    bm25_model: object,
    docs: List[Dict],
    queries: List[Dict],
    k: int,
    text_field: str = "content",
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Retrieve the top-k documents for each query with BM25.

    Parameters
    ----------
    bm25_model : object
        Fitted BM25 model.
    docs : list[dict]
        Documents aligned with the BM25 index.
    queries : list[dict]
        Query records.
    k : int
        Number of documents to return per query.
    text_field : str, default="content"
        Field used as query text.

    Returns
    -------
    tuple[np.ndarray, np.ndarray]
        Retrieved indices and BM25 scores.
    """
    if k <= 0:
        raise ValueError("k must be a positive integer.")

    n_docs = len(docs)
    if k > n_docs:
        raise ValueError("k cannot be greater than the number of documents.")

    n_queries = len(queries)
    if n_queries == 0:
        return (
            np.empty((0, k), dtype=np.int64),
            np.empty((0, k), dtype=np.float64),
        )

    topk_indices = np.zeros((n_queries, k), dtype=int)
    topk_scores = np.zeros((n_queries, k), dtype=float)

    for i, query in enumerate(tqdm(queries, desc="BM25 retrieval")):
        tokenized_query = tokenize(query[text_field])
        scores = np.array(bm25_model.get_scores(tokenized_query))
        topk = np.argsort(scores)[::-1][:k]

        topk_indices[i, : len(topk)] = topk
        topk_scores[i, : len(topk)] = scores[topk]

    return topk_indices, topk_scores


def map_indices_to_docids(topk_indices: np.ndarray, docs: List[dict]) -> List[List[str]]:
    """
    Verbatim copy of src/retrieval/bm25.py — converts indices → doc IDs.
    """
    return [[str(docs[int(idx)]["id"]) for idx in row] for row in topk_indices]

## Embeddings

Verbatim copy of `src/retrieval/embeddings.py`

In [ ]:
# === Verbatim copy of src/retrieval/embeddings.py ===

_MODEL_CACHE: dict[str, Any] = {}


def _normalize_texts(texts: list[str] | None) -> list[str]:
    """Normalize a text list and replace missing values with empty strings."""
    if texts is None:
        raise ValueError("texts must not be None.")
    return ["" if t is None else str(t) for t in texts]


def _texts_fingerprint(texts: list[str]) -> str:
    """Build a short fingerprint for a list of texts."""
    hasher = hashlib.sha256()
    for text in texts:
        encoded = text.encode("utf-8")
        hasher.update(len(encoded).to_bytes(8, byteorder="little"))
        hasher.update(encoded)
    return hasher.hexdigest()[:12]


def _looks_like_offline_or_cache_error(exc: Exception) -> bool:
    """
    Detect likely offline or missing-cache model loading errors.
    """
    message = str(exc).lower()
    exc_name = exc.__class__.__name__.lower()

    message_markers = (
        "nodename nor servname provided",
        "temporary failure in name resolution",
        "name resolution",
        "cannot send a request",
        "connection error",
        "max retries exceeded",
        "failed to establish a new connection",
        "httpsconnectionpool",
        "connecttimeout",
        "readtimeout",
        "offline",
        "could not connect",
        "not found in local cache",
        "is not a local folder",
        "repository not found",
    )
    name_markers = (
        "localentrynotfound",
        "repositorynotfound",
        "connectionerror",
        "connecttimeout",
        "readtimeout",
    )

    return any(marker in message for marker in message_markers) or any(
        marker in exc_name for marker in name_markers
    )


def _model_load_error_message(model_name: str, original_error: Exception) -> str:
    return (
        f"Failed to load SentenceTransformer model '{model_name}'. "
        "Likely cause: no network access and/or model not cached locally.\n"
        "How to fix:\n"
        "1) Preload the model once in an online environment:\n"
        "   from sentence_transformers import SentenceTransformer; "
        f"SentenceTransformer('{model_name}')\n"
        "2) Reuse the same local Hugging Face cache on this machine.\n"
        "3) Or reuse precomputed embeddings from data/cache for this dataset/model.\n"
        f"Original error: {original_error}"
    )


def _model_cache_key(
    model_name: str,
    device: str | None,
    backend: Literal["torch", "onnx", "openvino"],
    truncate_dim: int | None,
    model_max_seq_length: int | None,
    local_files_only: bool,
) -> str:
    return (
        f"{model_name}|device={device or 'auto'}|backend={backend}|"
        f"truncate_dim={truncate_dim}|max_seq={model_max_seq_length}|local={int(local_files_only)}"
    )


def _get_model(
    model_name: str,
    device: str | None = None,
    backend: Literal["torch", "onnx", "openvino"] = "torch",
    truncate_dim: int | None = None,
    model_max_seq_length: int | None = None,
    local_files_only: bool = False,
) -> "SentenceTransformer":
    """Load and cache a SentenceTransformer model instance."""
    try:
        from sentence_transformers import SentenceTransformer
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "sentence-transformers is required for encode_texts/build_embeddings. "
            "Install dependencies from requirements.txt."
        ) from exc

    cache_key = _model_cache_key(
        model_name=model_name,
        device=device,
        backend=backend,
        truncate_dim=truncate_dim,
        model_max_seq_length=model_max_seq_length,
        local_files_only=local_files_only,
    )
    if cache_key not in _MODEL_CACHE:
        try:
            model = SentenceTransformer(
                model_name,
                device=device,
                truncate_dim=truncate_dim,
                backend=backend,
                local_files_only=local_files_only,
            )
            if model_max_seq_length is not None:
                model.max_seq_length = int(model_max_seq_length)
            _MODEL_CACHE[cache_key] = model
        except Exception as exc:
            if _looks_like_offline_or_cache_error(exc):
                raise RuntimeError(_model_load_error_message(model_name, exc)) from exc
            raise
    return _MODEL_CACHE[cache_key]


def encode_texts(
    model_name: str,
    texts: list[str],
    batch_size: int = 64,
    show_progress_bar: bool = True,
    device: str | None = None,
    precision: Literal["float32", "int8", "uint8", "binary", "ubinary"] = "float32",
    model_max_seq_length: int | None = None,
    truncate_dim: int | None = None,
    chunk_size: int | None = None,
    normalize_embeddings: bool = True,
    backend: Literal["torch", "onnx", "openvino"] = "torch",
    local_files_only: bool = False,
) -> np.ndarray:
    """Encode texts into dense embeddings."""
    if batch_size <= 0:
        raise ValueError("batch_size must be a positive integer.")

    clean_texts = _normalize_texts(texts)
    model = _get_model(
        model_name=model_name,
        device=device,
        backend=backend,
        truncate_dim=truncate_dim,
        model_max_seq_length=model_max_seq_length,
        local_files_only=local_files_only,
    )

    emb = model.encode(
        clean_texts,
        batch_size=batch_size,
        show_progress_bar=show_progress_bar,
        precision=precision,
        convert_to_numpy=True,
        device=device,
        normalize_embeddings=normalize_embeddings,
        truncate_dim=truncate_dim,
        chunk_size=chunk_size,
    )

    emb = np.asarray(emb)
    if emb.ndim == 1:
        emb = emb.reshape(1, -1)

    return emb


def cache_save(path: Path, array: np.ndarray) -> None:
    """Save a numpy array to disk as `.npy`."""
    if path.suffix != ".npy":
        path = path.with_suffix(".npy")

    path.parent.mkdir(parents=True, exist_ok=True)
    np.save(path, array)


def cache_load(path: Path) -> np.ndarray:
    """Load a numpy array from disk."""
    if path.suffix != ".npy":
        path = path.with_suffix(".npy")

    if not path.exists():
        raise FileNotFoundError(f"Cache file not found: {path}")

    return np.load(path, allow_pickle=False)


def build_embeddings(
    docs: list[dict],
    queries: list[dict],
    text_field: str = "content",
    model_name: str = "all-MiniLM-L6-v2",
    batch_size: int = 64,
    show_progress_bar: bool = True,
    cache_dir: Path = Path("data/cache"),
    device: str | None = None,
    precision: Literal["float32", "int8", "uint8", "binary", "ubinary"] = "float32",
    truncate_dim: int | None = None,
    chunk_size: int | None = None,
    normalize_embeddings: bool = True,
    backend: Literal["torch", "onnx", "openvino"] = "torch",
    model_max_seq_length: int | None = None,
    local_files_only: bool = False,
    use_cache: bool = True,
) -> tuple[np.ndarray, np.ndarray]:
    """Build document and query embeddings, optionally using the cache."""

    if use_cache:
        cache_dir.mkdir(parents=True, exist_ok=True)

    doc_texts = [str(d.get(text_field) or "") for d in docs]
    query_texts = [str(q.get(text_field) or "") for q in queries]

    safe_model_name = model_name.replace("/", "_")
    safe_text_field = text_field.replace("/", "_")
    safe_precision = precision.replace("/", "_")
    norm_tag = "norm1" if normalize_embeddings else "norm0"
    trunc_tag = f"trunc{truncate_dim}" if truncate_dim is not None else "truncNone"
    maxseq_tag = f"maxseq{model_max_seq_length}" if model_max_seq_length is not None else "maxseqNone"
    device_tag = (device or "auto").replace("/", "_")
    backend_tag = backend.replace("/", "_")
    local_tag = f"local{int(local_files_only)}"
    doc_fingerprint = _texts_fingerprint(doc_texts)
    query_fingerprint = _texts_fingerprint(query_texts)
    doc_cache_path = cache_dir / (
        f"docs_{safe_model_name}_{safe_text_field}_{safe_precision}_{norm_tag}_"
        f"{trunc_tag}_{maxseq_tag}_{backend_tag}_{device_tag}_{local_tag}_{doc_fingerprint}.npy"
    )
    query_cache_path = cache_dir / (
        f"queries_{safe_model_name}_{safe_text_field}_{safe_precision}_{norm_tag}_"
        f"{trunc_tag}_{maxseq_tag}_{backend_tag}_{device_tag}_{local_tag}_{query_fingerprint}.npy"
    )

    if use_cache and doc_cache_path.exists():
        doc_embeddings = cache_load(doc_cache_path)
    else:
        doc_embeddings = encode_texts(
            model_name,
            doc_texts,
            batch_size=batch_size,
            show_progress_bar=show_progress_bar,
            device=device,
            precision=precision,
            model_max_seq_length=model_max_seq_length,
            truncate_dim=truncate_dim,
            chunk_size=chunk_size,
            normalize_embeddings=normalize_embeddings,
            backend=backend,
            local_files_only=local_files_only,
        )
        if use_cache:
            cache_save(doc_cache_path, doc_embeddings)

    if use_cache and query_cache_path.exists():
        query_embeddings = cache_load(query_cache_path)
    else:
        query_embeddings = encode_texts(
            model_name,
            query_texts,
            batch_size=batch_size,
            show_progress_bar=show_progress_bar,
            device=device,
            precision=precision,
            model_max_seq_length=model_max_seq_length,
            truncate_dim=truncate_dim,
            chunk_size=chunk_size,
            normalize_embeddings=normalize_embeddings,
            backend=backend,
            local_files_only=local_files_only,
        )
        if use_cache:
            cache_save(query_cache_path, query_embeddings)

    return doc_embeddings, query_embeddings


def _are_rows_unit_norm(
    x: np.ndarray,
    *,
    atol: float = 2e-2,
    max_rows: int = 256,
) -> bool:
    """Return whether the sampled rows look unit-normalized."""
    if x.ndim != 2 or x.shape[0] == 0:
        return False

    sample = x if x.shape[0] <= max_rows else x[:max_rows]
    norms = np.linalg.norm(sample, axis=1)
    return np.all(np.isfinite(norms)) and np.allclose(norms, 1.0, atol=atol)


def retrieve_embeddings(
    doc_emb: np.ndarray,
    query_emb: np.ndarray,
    k: int,
    assume_normalized: bool = True,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Retrieve the top-k documents per query with cosine similarity.
    """
    if k <= 0:
        raise ValueError("k must be a positive integer.")

    doc_emb = np.asarray(doc_emb, dtype=np.float32)
    query_emb = np.asarray(query_emb, dtype=np.float32)

    if doc_emb.ndim != 2 or query_emb.ndim != 2:
        raise ValueError("doc_emb and query_emb must be 2D arrays.")
    if doc_emb.shape[1] != query_emb.shape[1]:
        raise ValueError("doc_emb and query_emb must have the same embedding dimension.")
    if k > doc_emb.shape[0]:
        raise ValueError("k cannot be greater than the number of documents.")

    n_queries = query_emb.shape[0]
    if n_queries == 0:
        return (
            np.empty((0, k), dtype=np.int64),
            np.empty((0, k), dtype=np.float32),
        )

    if assume_normalized and _are_rows_unit_norm(doc_emb) and _are_rows_unit_norm(query_emb):
        similarities = query_emb @ doc_emb.T
    else:
        doc_norm = np.linalg.norm(doc_emb, axis=1, keepdims=True)
        query_norm = np.linalg.norm(query_emb, axis=1, keepdims=True)
        doc_unit = doc_emb / np.clip(doc_norm, 1e-12, None)
        query_unit = query_emb / np.clip(query_norm, 1e-12, None)
        similarities = query_unit @ doc_unit.T

    topk_unsorted_idx = np.argpartition(-similarities, kth=k - 1, axis=1)[:, :k]
    topk_unsorted_scores = np.take_along_axis(similarities, topk_unsorted_idx, axis=1)
    rerank = np.argsort(-topk_unsorted_scores, axis=1)

    topk_indices = np.take_along_axis(topk_unsorted_idx, rerank, axis=1)
    topk_scores = np.take_along_axis(topk_unsorted_scores, rerank, axis=1)

    return topk_indices, topk_scores

## Hybrid Retrieval

Verbatim copy of `src/retrieval/hybrid.py`

In [ ]:
# === Verbatim copy of src/retrieval/hybrid.py ===

def truncate_text_field(
    items: list[dict],
    text_field: str,
    max_chars: int | None,
) -> list[dict]:
    """
    Return copies of items with a truncated text field when requested.
    """
    if max_chars is None:
        return items
    if max_chars <= 0:
        raise ValueError("max_chars must be positive when provided.")

    truncated_items: list[dict] = []
    for item in items:
        updated = item.copy()
        updated[text_field] = str(updated.get(text_field) or "")[:max_chars]
        truncated_items.append(updated)
    return truncated_items


def fuse_rankings_rrf(
    emb_indices: np.ndarray,
    bm25_indices: np.ndarray,
    k_out: int,
    rrf_k: int = 60,
    weight_embeddings: float = 3.5,
    weight_bm25: float = 0.5,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Fuse embedding and BM25 rankings with weighted Reciprocal Rank Fusion.
    """
    if k_out <= 0:
        raise ValueError("k_out must be > 0")
    if rrf_k <= 0:
        raise ValueError("rrf_k must be > 0")
    if weight_embeddings <= 0 or weight_bm25 <= 0:
        raise ValueError("RRF weights must be > 0")
    if emb_indices.ndim != 2 or bm25_indices.ndim != 2:
        raise ValueError("emb_indices and bm25_indices must be 2D arrays")
    if emb_indices.shape[0] != bm25_indices.shape[0]:
        raise ValueError("emb_indices and bm25_indices must have the same number of queries")

    n_queries = emb_indices.shape[0]
    fused_indices = np.zeros((n_queries, k_out), dtype=np.int64)
    fused_scores = np.zeros((n_queries, k_out), dtype=np.float32)

    for qi in range(n_queries):
        scores: dict[int, float] = {}

        for rank, doc_idx in enumerate(emb_indices[qi], start=1):
            did = int(doc_idx)
            scores[did] = scores.get(did, 0.0) + (weight_embeddings / (rrf_k + rank))

        for rank, doc_idx in enumerate(bm25_indices[qi], start=1):
            did = int(doc_idx)
            scores[did] = scores.get(did, 0.0) + (weight_bm25 / (rrf_k + rank))

        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:k_out]
        fused_indices[qi, : len(ranked)] = [doc_id for doc_id, _ in ranked]
        fused_scores[qi, : len(ranked)] = [float(score) for _, score in ranked]

    return fused_indices, fused_scores


def retrieve_hybrid_bm25_embeddings(
    docs: list[dict],
    queries: list[dict],
    *,
    top_k: int,
    text_field: str = "content",
    embedding_model_name: str = "all-MiniLM-L12-v2",
    embedding_batch_size: int = 64,
    show_progress_bar: bool = True,
    cache_dir: Path = Path("data/cache"),
    embedding_device: str | None = None,
    embedding_precision: Literal["float32", "int8", "uint8", "binary", "ubinary"] = "float32",
    embedding_max_seq_length: int | None = None,
    embedding_truncate_dim: int | None = None,
    embedding_chunk_size: int | None = None,
    embedding_normalize: bool = True,
    embedding_backend: Literal["torch", "onnx", "openvino"] = "torch",
    embedding_local_files_only: bool = False,
    hybrid_bm25_method: str = "plus",
    hybrid_candidate_multiplier: int = 1,
    hybrid_rrf_k: int = 60,
    hybrid_weight_embeddings: float = 3.5,
    hybrid_weight_bm25: float = 0.5,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Retrieve documents with BM25 and embeddings, then fuse both rankings.
    """
    if top_k <= 0:
        raise ValueError("top_k must be a positive integer.")
    if top_k > len(docs):
        raise ValueError(f"top_k={top_k} cannot be greater than number of docs={len(docs)}")
    if hybrid_candidate_multiplier <= 0:
        raise ValueError("hybrid_candidate_multiplier must be > 0")

    candidate_k = min(len(docs), max(top_k, top_k * int(hybrid_candidate_multiplier)))

    bm25_model = fit_bm25(docs, text_field=text_field, method=hybrid_bm25_method)
    bm25_indices, _ = retrieve_bm25(
        bm25_model,
        docs,
        queries,
        k=candidate_k,
        text_field=text_field,
    )

    doc_emb, query_emb = build_embeddings(
        docs,
        queries,
        text_field=text_field,
        model_name=embedding_model_name,
        batch_size=embedding_batch_size,
        show_progress_bar=show_progress_bar,
        cache_dir=cache_dir,
        device=embedding_device,
        precision=embedding_precision,
        model_max_seq_length=embedding_max_seq_length,
        truncate_dim=embedding_truncate_dim,
        chunk_size=embedding_chunk_size,
        normalize_embeddings=embedding_normalize,
        backend=embedding_backend,
        local_files_only=embedding_local_files_only,
    )

    emb_indices, _ = retrieve_embeddings(
        doc_emb,
        query_emb,
        k=candidate_k,
        assume_normalized=embedding_normalize,
    )

    return fuse_rankings_rrf(
        emb_indices=emb_indices,
        bm25_indices=bm25_indices,
        k_out=top_k,
        rrf_k=hybrid_rrf_k,
        weight_embeddings=hybrid_weight_embeddings,
        weight_bm25=hybrid_weight_bm25,
    )

## Submission Formatting

Verbatim copy of `src/kaggle/format.py`

In [ ]:
# === Verbatim copy of src/kaggle/format.py ===

SUBMISSION_COLUMNS = ["query_id", "relevant_doc_ids", "category"]
DEFAULT_CATEGORY = "?"
DEFAULT_TOP_K = 100


def _normalize_docids(docids: list[str], top_k: int) -> list[str]:
    if top_k <= 0:
        raise ValueError("top_k must be a positive integer.")

    seen: set[str] = set()
    normalized: list[str] = []
    for doc_id in docids:
        doc_id_str = str(doc_id)
        if doc_id_str not in seen:
            seen.add(doc_id_str)
            normalized.append(doc_id_str)

    if len(normalized) < top_k:
        raise ValueError(
            f"Each query must have at least {top_k} predicted doc IDs, got {len(normalized)}."
        )

    return normalized[:top_k]


def make_submission(
    query_ids: list[str],
    pred_docids: list[list[str]],
    top_k: int = DEFAULT_TOP_K,
    category: str = DEFAULT_CATEGORY,
    categories: dict[str, str] | None = None,
):
    """
    Build a Kaggle submission DataFrame.

    Parameters
    ----------
    query_ids : list[str]
        Ordered query identifiers.
    pred_docids : list[list[str]]
        Predicted document IDs aligned with `query_ids`.
    top_k : int, default=100
        Number of document IDs kept per query.
    category : str, default="?"
        Fallback category value used when `categories` is not provided.
    categories : dict[str, str] | None, default=None
        Optional per-query predicted categories for Phase 2.
    """
    if len(query_ids) != len(pred_docids):
        raise ValueError("query_ids and pred_docids must have the same length.")

    rows: list[dict[str, str]] = []
    for i, (qid, docs_for_query) in enumerate(zip(query_ids, pred_docids)):
        if docs_for_query is None:
            raise ValueError(f"pred_docids[{i}] is None.")

        normalized_docids = _normalize_docids(list(docs_for_query), top_k=top_k)

        row_category = categories.get(str(qid), category) if categories else category

        rows.append(
            {
                "query_id": str(qid),
                "relevant_doc_ids": json.dumps(normalized_docids, ensure_ascii=False),
                "category": str(row_category),
            }
        )

    submission_df = pd.DataFrame(rows, columns=SUBMISSION_COLUMNS)
    return submission_df


def save_submission(
    query_ids: list[str],
    pred_docids: list[list[str]],
    output_path: Path = Path("outputs/submissions/submission.csv"),
    top_k: int = DEFAULT_TOP_K,
    category: str = DEFAULT_CATEGORY,
    categories: dict[str, str] | None = None,
):
    """
    Create and save a Kaggle submission CSV.
    """
    submission_df = make_submission(
        query_ids=query_ids,
        pred_docids=pred_docids,
        top_k=top_k,
        category=category,
        categories=categories,
    )

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    submission_df.to_csv(output_path, index=False)
    return submission_df

## Evaluation (partial)

Verbatim copy of `src/evaluation/evaluate.py` — `indices_to_docids` function only

In [ ]:
# === Verbatim copy of src/evaluation/evaluate.py (indices_to_docids only) ===

def indices_to_docids(
    topk_indices: ArrayLike,
    docs: Sequence[dict],
    doc_id_field: str = "id",
) -> list[list[str]]:
    """
    Convert retrieved document indices into document IDs.
    """
    if not docs:
        raise ValueError("docs must not be empty.")

    topk_indices_arr = np.asarray(topk_indices)
    if topk_indices_arr.ndim != 2:
        raise ValueError(
            "topk_indices must be a 2D array-like structure with shape [n_queries, k]."
        )

    doc_ids = []
    for i, doc in enumerate(docs):
        if doc_id_field not in doc:
            raise KeyError(f"docs[{i}] does not contain '{doc_id_field}'.")
        doc_ids.append(str(doc[doc_id_field]))

    pred_docids: list[list[str]] = []
    n_docs = len(doc_ids)
    for q_idx, row in enumerate(topk_indices_arr):
        row_docids: list[str] = []
        for rank_idx, doc_idx in enumerate(row):
            idx = int(doc_idx)
            if idx < 0 or idx >= n_docs:
                raise IndexError(
                    f"topk_indices[{q_idx}][{rank_idx}]={idx} is out of bounds for {n_docs} documents."
                )
            row_docids.append(doc_ids[idx])
        pred_docids.append(row_docids)

    return pred_docids

## Feature Extraction

Verbatim copy of `src/classification/features.py`

In [ ]:
# === Verbatim copy of src/classification/features.py ===

from sklearn.model_selection import train_test_split

_DEFAULT_EMBEDDING_MODEL = "all-MiniLM-L12-v2"
_DEFAULT_EMBEDDING_BATCH_SIZE = 64


def extract_texts_and_labels(
    docs: list[dict],
    queries: list[dict] | None = None,
    text_field: str = "content",
) -> tuple[list[str], list[str]]:
    """
    Extract texts and labels from documents and optional queries.

    Parameters
    ----------
    docs : list[dict]
        Documents containing `text_field` and `category`.
    queries : list[dict] | None, default=None
        Optional query records to append.
    text_field : str, default="content"
        Field used as input text.

    Returns
    -------
    tuple[list[str], list[str]]
        Texts and aligned category labels.
    """
    texts = []
    labels = []
    for doc in docs:
        text = doc.get(text_field)
        label = doc.get("category")
        texts.append(text)
        labels.append(label)
    if queries:
        for query in queries:
            text = query.get(text_field)
            label = query.get("category")
            texts.append(text)
            labels.append(label)

    return texts, labels


def build_tfidf_features(
    X_train: list[str],
    X_val: list[str] | None = None,
    X_test: list[str] | None = None,
    max_features: int = 50_000,
    ngram_range: tuple[int, int] = (1, 2),
    sublinear_tf: bool = True,
) -> tuple:
    """
    Build TF-IDF feature matrices from text inputs.

    Parameters
    ----------
    X_train : list[str]
        Training texts.
    X_val : list[str] | None, default=None
        Optional validation texts.
    X_test : list[str] | None, default=None
        Optional test texts.
    max_features : int, default=50_000
        Maximum vocabulary size.
    ngram_range : tuple[int, int], default=(1, 2)
        N-gram range passed to `TfidfVectorizer`.
    sublinear_tf : bool, default=True
        Whether to use sublinear term-frequency scaling.

    Returns
    -------
    tuple
        Tuple starting with `(vectorizer, X_train_mat)` and extended with
        transformed validation and test matrices when provided.
    """
    vec = TfidfVectorizer(
        max_features=max_features,
        ngram_range=ngram_range,
        sublinear_tf=sublinear_tf
    )

    X_train_mat = vec.fit_transform(X_train)
    result = (vec, X_train_mat)

    if X_val is not None:
        X_val_mat = vec.transform(X_val)
        result += (X_val_mat,)

    if X_test is not None:
        X_test_mat = vec.transform(X_test)
        result += (X_test_mat,)

    return result


def build_count_features(
    X_train: list[str],
    X_val: list[str] | None = None,
    X_test: list[str] | None = None,
    max_features: int = 50_000,
    ngram_range: tuple[int, int] = (1, 2),
) -> tuple:
    """
    Build count-based feature matrices from text inputs.
    """
    vec = CountVectorizer(
        max_features=max_features,
        ngram_range=ngram_range
    )

    X_train_mat = vec.fit_transform(X_train)
    result = (vec, X_train_mat)

    if X_val is not None:
        X_val_mat = vec.transform(X_val)
        result += (X_val_mat,)

    if X_test is not None:
        X_test_mat = vec.transform(X_test)
        result += (X_test_mat,)

    return result


def build_embedding_features(
    X_train: list[str],
    X_val: list[str] | None = None,
    X_test: list[str] | None = None,
    model_name: str = _DEFAULT_EMBEDDING_MODEL,
    batch_size: int = _DEFAULT_EMBEDDING_BATCH_SIZE,
) -> tuple:
    """
    Build dense embedding matrices from text inputs.
    """
    X_train_mat = encode_texts(
        model_name=model_name,
        texts=X_train,
        batch_size=batch_size,
        device=None
    )
    result = (None, X_train_mat)

    if X_val is not None:
        X_val_mat = encode_texts(
            model_name=model_name,
            texts=X_val,
            batch_size=batch_size,
            device=None
        )
        result += (X_val_mat,)

    if X_test is not None:
        X_test_mat = encode_texts(
            model_name=model_name,
            texts=X_test,
            batch_size=batch_size,
            device=None
        )
        result += (X_test_mat,)

    return result

## Classifier

Verbatim copy of `src/classification/model.py`

In [ ]:
# === Verbatim copy of src/classification/model.py ===

class Classifier:
    """
    Unified wrapper around the supported sklearn classifiers.

    Parameters
    ----------
    method : str, default="logreg"
        One of `"nb"`, `"svc"`, `"logreg"`, or `"mlp"`.
    **kwargs
        Extra keyword arguments forwarded to the underlying estimator.
    """

    SUPPORTED_METHODS = ("nb", "svc", "logreg", "mlp")

    def __init__(self, method: str = "logreg", **kwargs) -> None:
        if method not in self.SUPPORTED_METHODS:
            raise ValueError(
                f"method must be one of {self.SUPPORTED_METHODS}, got {method!r}."
            )
        self.method = method
        self._kwargs = kwargs
        self._model = None
        self._classes: list[str] = []

    @property
    def classes_(self) -> list[str]:
        """Ordered category labels, aligned with predict_proba columns."""
        if not self._classes:
            raise RuntimeError("Classifier has not been fitted yet. Call fit() first.")
        return self._classes

    def fit(self, X, y: list[str]) -> None:
        """
        Fit the classifier on a feature matrix and label vector.
        """
        kwargs = self._kwargs.copy()
        if self.method in ("svc", "logreg", "mlp"):
            kwargs.setdefault("random_state", RANDOM_SEED)

        if self.method == "nb":
            self._model = MultinomialNB(**kwargs)
        elif self.method == "svc":
            self._model = LinearSVC(**kwargs)
        elif self.method == "logreg":
            self._model = LogisticRegression(**kwargs)
        elif self.method == "mlp":
            self._model = MLPClassifier(**kwargs)

        self._model.fit(X, y)
        self._classes = self._model.classes_.tolist()

    def predict(self, X) -> list[str]:
        """
        Predict category labels for the input samples.
        """
        if self._model is None:
            raise RuntimeError("Classifier has not been fitted yet. Call fit() first.")
        return self._model.predict(X).tolist()

    def predict_proba(self, X) -> np.ndarray | None:
        """
        Predict class probabilities when the underlying model supports them.
        """
        if self._model is None:
            raise RuntimeError("Classifier has not been fitted yet. Call fit() first.")
        if self.method != "svc":
            return self._model.predict_proba(X)
        return None

    def __repr__(self) -> str:
        fitted = self._model is not None
        return f"Classifier(method={self.method!r}, fitted={fitted})"

## Reranking

Verbatim copy of `src/classification/rerank.py`

In [ ]:
# === Verbatim copy of src/classification/rerank.py ===

def hard_filter(
    topk_doc_ids: list[str],
    topk_scores: np.ndarray,
    docs_by_id: dict[str, dict],
    predicted_category: str,
    fallback_to_original: bool = True,
) -> tuple[list[str], np.ndarray]:
    """
    Move category-matching documents to the front of the ranking.
    """

    matched_ids = []
    matched_scores = []
    excluded_ids = []
    excluded_scores = []

    for i in range(len(topk_doc_ids)):
        doc_id = topk_doc_ids[i]
        score = topk_scores[i]
        doc = docs_by_id[doc_id]

        if doc["category"] == predicted_category:
            matched_ids.append(doc_id)
            matched_scores.append(score)
        else:
            excluded_ids.append(doc_id)
            excluded_scores.append(score)

    if fallback_to_original and len(matched_ids) < len(topk_doc_ids):
        matched_ids += excluded_ids
        matched_scores += excluded_scores

    return matched_ids, np.array(matched_scores)


def soft_boost(
    topk_doc_ids: list[str],
    topk_scores: np.ndarray,
    docs_by_id: dict[str, dict],
    category_proba: np.ndarray | None,
    classes: list[str],
    boost_factor: float = 1.5,
) -> tuple[list[str], np.ndarray]:
    """
    Re-score documents with the predicted category probabilities.
    """

    if category_proba is None:
        return topk_doc_ids, topk_scores

    prob_by_cat = {}
    for i in range(len(classes)):
        prob_by_cat[classes[i]] = category_proba[i]

    boosted_scores = []
    for i in range(len(topk_doc_ids)):
        doc_id = topk_doc_ids[i]
        score = topk_scores[i]
        doc_category = docs_by_id[doc_id]["category"]
        proba = prob_by_cat.get(doc_category, 0.0)
        new_score = score * (1 + boost_factor * proba)
        boosted_scores.append(new_score)

    boosted_scores = np.array(boosted_scores)
    sorted_indices = np.argsort(boosted_scores)[::-1]

    reranked_doc_ids = [topk_doc_ids[i] for i in sorted_indices]
    reranked_scores = boosted_scores[sorted_indices]

    return reranked_doc_ids, reranked_scores


def build_docs_index(docs: list[dict]) -> dict[str, dict]:
    """
    Build a dictionary keyed by document ID for fast lookup.
    """
    return {str(doc["id"]): doc for doc in docs}

## Phase 2 Pipeline

Verbatim copy of `src/kaggle/submit_phase2.py` — adapted to use this notebook's global variables.

In [ ]:
# === Verbatim copy of src/kaggle/submit_phase2.py ===

_CATEGORY_EXPANSIONS = {
    "android":     "android mobile app development java kotlin apk",
    "tex":         "tex latex document typesetting mathematics formula",
    "unix":        "unix linux shell bash terminal command line",
    "gaming":      "gaming video game console steam multiplayer",
    "programmers": "programming software development code algorithm design",
}


def set_global_seeds(seed=RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


def run_phase2_pipeline(
    raw_dir=RAW_DIR,
    cache_dir=CACHEDIR,
    output_submission=OUTPUT_SUBMISSION,
    classifier_method=CLASSIFIER_METHOD,
    feature_method=FEATURE_METHOD,
    rerank_strategy=RERANK_STRATEGY,
    rerank_boost_factor=RERANK_BOOST_FACTOR,
    top_k=TOP_K,
    text_field=TEXT_FIELD,
    retrieval_k=RETRIEVAL_K,
    prf_top_k=PRF_TOP_K,
    prf_max_tags=PRF_MAX_TAGS,
    embedding_model_name=EMBEDDING_MODEL_NAME,
    hybrid_rrf_k=HYBRID_RRF_K,
    hybrid_weight_embeddings=HYBRID_WEIGHT_EMBEDDINGS,
    hybrid_weight_bm25=HYBRID_WEIGHT_BM25,
    random_seed=RANDOM_SEED,
):
    set_global_seeds(random_seed)

    # 1. Load data (always from raw to guarantee consistency)
    docs          = load_json(raw_dir / "docs.json")
    queries_train = load_json(raw_dir / "queries_train.json")
    queries_test  = load_json(raw_dir / "queries_test.json")

    docs, queries_test = add_content_field(docs, queries_test, clean=False)
    _, queries_train   = add_content_field([], queries_train, clean=False)
    print(f"Docs: {len(docs)} | Train: {len(queries_train)} | Test: {len(queries_test)}")

    # 2. Features + classifier
    all_train_texts, all_train_labels = extract_texts_and_labels(docs, queries_train, text_field=text_field)
    test_texts, _                     = extract_texts_and_labels(queries_test, text_field=text_field)

    if feature_method == "tfidf":
        _, X_train, X_test = build_tfidf_features(all_train_texts, X_val=None, X_test=test_texts)
    elif feature_method == "count":
        _, X_train, X_test = build_count_features(all_train_texts, X_val=None, X_test=test_texts)
    elif feature_method == "embeddings":
        _, X_train, X_test = build_embedding_features(all_train_texts, X_val=None, X_test=test_texts)
    else:
        raise ValueError(f"feature_method inconnu : {feature_method!r}")

    clf = Classifier(method=classifier_method)
    clf.fit(X_train, all_train_labels)

    # 3. Category prediction
    predicted_categories = clf.predict(X_test)
    print(f"Classifieur: {classifier_method} | classes: {clf._classes}")

    # 4. Query expansion
    queries_expanded = []
    for q, cat in zip(queries_test, predicted_categories):
        q_exp = q.copy()
        q_exp["content"] = q["content"] + " " + _CATEGORY_EXPANSIONS.get(cat, cat)
        queries_expanded.append(q_exp)

    # 5. PRF : BM25 top-k -> tags -> append
    print("PRF : fitting BM25...")
    cache_dir.mkdir(parents=True, exist_ok=True)
    bm25_prf = fit_bm25(docs, text_field=text_field)
    prf_indices, _ = retrieve_bm25(bm25_prf, docs, queries_expanded, k=prf_top_k, text_field=text_field)

    queries_final = []
    for i, q_exp in enumerate(queries_expanded):
        tags_list = []
        for doc_idx in prf_indices[i]:
            doc_tags = docs[doc_idx].get("tags", [])
            if isinstance(doc_tags, list):
                tags_list.extend(t for t in doc_tags if isinstance(t, str) and t.strip())
        unique_tags = list(dict.fromkeys(tags_list))[:prf_max_tags]
        q_final = q_exp.copy()
        if unique_tags:
            q_final["content"] = q_exp["content"] + " " + " ".join(unique_tags)
        queries_final.append(q_final)

    # 6. Hybrid retrieval BM25 + Embeddings (RRF)
    print(f"Retrieval: {retrieval_k} candidates per query (rrf_k={hybrid_rrf_k})...")
    topk_indices, topk_scores = retrieve_hybrid_bm25_embeddings(
        docs=docs,
        queries=queries_final,
        top_k=retrieval_k,
        text_field=text_field,
        cache_dir=cache_dir,
        embedding_model_name=embedding_model_name,
        hybrid_weight_embeddings=hybrid_weight_embeddings,
        hybrid_weight_bm25=hybrid_weight_bm25,
        hybrid_rrf_k=hybrid_rrf_k,
    )
    pred_docids = map_indices_to_docids(topk_indices, docs)

    # 7. Reranking
    category_proba = clf.predict_proba(X_test) if rerank_strategy == "soft_boost" else None
    docs_by_id = build_docs_index(docs)
    reranked_docids = []

    for i in tqdm(range(len(queries_test)), desc="Reranking"):
        if rerank_strategy == "hard_filter":
            reranked_ids, _ = hard_filter(
                topk_doc_ids=pred_docids[i],
                topk_scores=topk_scores[i],
                docs_by_id=docs_by_id,
                predicted_category=predicted_categories[i],
                fallback_to_original=True,
            )
        elif rerank_strategy == "soft_boost":
            reranked_ids, _ = soft_boost(
                topk_doc_ids=pred_docids[i],
                topk_scores=topk_scores[i],
                docs_by_id=docs_by_id,
                category_proba=category_proba[i],
                classes=clf._classes,
                boost_factor=rerank_boost_factor,
            )
        else:
            reranked_ids = pred_docids[i]
        reranked_docids.append(reranked_ids)

    # 8. Save submission
    categories_dict = {str(q["id"]): cat for q, cat in zip(queries_test, predicted_categories)}
    output_submission.parent.mkdir(parents=True, exist_ok=True)
    df = save_submission(
        query_ids=[str(q["id"]) for q in queries_test],
        pred_docids=reranked_docids,
        output_path=output_submission,
        top_k=top_k,
        categories=categories_dict,
    )
    print(f"\nSubmission saved: {output_submission}")
    print(f"Lignes : {len(df)} | Top-K : {top_k}")
    return df

## Pipeline Execution

Runs the full Phase 2 pipeline with the configuration defined above.

In [ ]:
set_global_seeds(RANDOM_SEED)
submission_df = run_phase2_pipeline()
submission_df.head()

## Validation and Summary

Verifies that the submission meets Kaggle expectations: 141 queries × 100 documents.

In [ ]:
import json as _json
assert len(submission_df) == 141, f"Expected 141 queries, got {len(submission_df)}"
_first = _json.loads(submission_df.iloc[0]["relevant_doc_ids"])
assert len(_first) == TOP_K, f"Expected {TOP_K} doc IDs, got {len(_first)}"
assert submission_df["category"].nunique() > 0
elapsed = time.perf_counter() - NOTEBOOK_T0
print(f"Validation passed — {len(submission_df)} queries x {TOP_K} docs")
print(f"Total time: {elapsed:.1f}s")
print(f"Submission : {OUTPUT_SUBMISSION}")
submission_df.head(3)